# SMS Spam Detection

This project compares five spam detection approaches across multiple datasets. In addition to evaluating models on their original datasets, the experiments investigate cross-dataset generalization, training-data size, realistic class imbalance, inference speed, model size, and classification errors.

Four datasets are used: UCI SMS Spam Collection, Turkish SMS Collection, YouTube Spam Collection, and Enron Spam. The SMS datasets represent the primary task domain, while YouTube comments and Enron emails introduce different text domains to evaluate how well the models generalize beyond the type of data on which they are trained.

Reusable functionality for dataset loading, preprocessing, metrics, experiments, and model implementations is kept in the `src/` directory. This notebook documents the experimental workflow, including dataset validation, model development, evaluation, and visualization of the results.

In [1]:
import sys
from pathlib import Path

import pandas as pd

# Add the project root so modules under src/ can be imported.
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

## 0. Google Colab Environment Setup

The project is run in Google Colab so that the transformer-based models can use a GPU during fine-tuning, as recommended in the assignment.

The project repository is cloned into the Colab runtime, the GPU environment is verified, and the raw datasets are restored from Google Drive before running the data preparation and experiments.

In [2]:
import os

# Clone the repository only if it is not already present.
if not os.path.exists("/content/spam-detection"):
    !git clone https://github.com/ddolapcioglu24/spam-detection.git

# Move into the project directory.
%cd /content/spam-detection

/content/spam-detection


### 0.1 GPU Verification

The Colab runtime is configured with an NVIDIA T4 GPU. CUDA availability is checked to verify that PyTorch can access the GPU before transformer fine-tuning begins.

In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: False


### 0.2 Python Environment Check

The main Python packages used by the project are checked to confirm that the Colab environment can load the libraries required by the data pipeline, transformer models, SVM, and fastText implementation.

In [4]:
import numpy as np
import pandas as pd
import sklearn
import transformers
import datasets
import accelerate
import fasttext

print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("accelerate:", accelerate.__version__)
print("fastText imported successfully")

NumPy: 2.0.2
pandas: 2.2.2
scikit-learn: 1.6.1
PyTorch: 2.11.0+cpu
transformers: 5.13.1
datasets: 4.0.0
accelerate: 1.14.0
fastText imported successfully


### 0.3 Dataset Setup

The raw datasets are excluded from the Git repository. To avoid uploading the same files again after every Colab runtime restart, the compressed raw dataset directory is stored persistently in Google Drive.

Google Drive is mounted in the Colab runtime, and the archive is extracted into the project directory so that the dataset paths match those defined in `src/config.py`.

In [5]:
from google.colab import drive

# Mount Google Drive to access the stored raw datasets.
drive.mount("/content/drive")

# Extract the raw datasets into the project directory.
!unzip -q "/content/drive/MyDrive/spam-detection/raw_data.zip" -d .

Mounted at /content/drive


## 1. Dataset Preparation and Validation

Four datasets from different languages and domains are used:

1. **UCI SMS Spam Collection** — English SMS messages.
2. **Turkish SMS Collection** — Turkish SMS messages.
3. **UCI YouTube Spam Collection** — YouTube comments from a different text domain.
4. **Enron Spam** — English emails from a different communication domain.

Before training any model, the datasets are inspected for their size, class distribution, missing values, internal duplicates, and overlap with the other datasets. Cross-dataset overlap is checked after normalizing the text by lowercasing, removing punctuation, and collapsing extra whitespace.

In [6]:
# Import reusable dataset utilities.
from src.data import (
    load_uci_sms,
    load_turkish_sms,
    load_youtube_spam,
    load_enron_spam,
    normalize_text,
    get_duplicate_stats,
    get_dataset_overlap,
)

### 1.1 Loading the Datasets

Each dataset has its own loading function in `src/data.py`. Although the original files have different formats and column names, every loader returns the same two-column representation:

- `label`: `ham` or `spam`
- `text`: the original message or comment

Using a common representation allows the same evaluation and modeling code to be reused across all four datasets.

In [7]:
# Load all four datasets using their dedicated loaders.
uci = load_uci_sms()
turkish = load_turkish_sms()
youtube = load_youtube_spam()
enron = load_enron_spam()

# Keep datasets together for reusable analysis.
datasets = {
    "UCI SMS": uci,
    "Turkish SMS": turkish,
    "YouTube Spam": youtube,
    "Enron Spam": enron,
}

# Preview the standardized UCI dataset.
uci.head()

,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


### 1.2 Dataset Summary

Before modeling, we verify the size and class distribution of each dataset. This is especially important because the proportion of spam messages differs substantially across the four datasets.

We also check for missing labels or text entries to make sure the datasets are ready for later experiments.

In [8]:
# Build a summary of size, class distribution, and missing values.
summary_rows = []

for name, df in datasets.items():
    spam_count = (df["label"] == "spam").sum()
    ham_count = (df["label"] == "ham").sum()

    summary_rows.append({
        "dataset": name,
        "total": len(df),
        "ham": ham_count,
        "spam": spam_count,
        "spam_ratio": spam_count / len(df),
        "missing_text": df["text"].isna().sum(),
        "missing_label": df["label"].isna().sum(),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,dataset,total,ham,spam,spam_ratio,missing_text,missing_label
0,UCI SMS,5572,4825,747,0.134063,0,0
1,Turkish SMS,4751,2215,2536,0.533782,0,0
2,YouTube Spam,1956,951,1005,0.513804,0,0
3,Enron Spam,33716,16545,17171,0.509283,0,0


### 1.3 Duplicate Analysis

Duplicate messages can affect the reliability of model evaluation, especially if identical messages appear in different datasets.

For duplicate detection, message text is normalized by converting it to lowercase, removing punctuation, and collapsing extra whitespace. We first measure repeated messages within each dataset and then check for overlap between different datasets.

In [9]:
# Calculate duplicate statistics within each dataset.
duplicate_rows = []

for name, df in datasets.items():
    stats = get_duplicate_stats(df)

    duplicate_rows.append({
        "dataset": name,
        **stats,
    })

duplicate_df = pd.DataFrame(duplicate_rows)
duplicate_df

,dataset,total_messages,unique_messages,duplicate_rows
0,UCI SMS,5572,5131,441
1,Turkish SMS,4751,4696,55
2,YouTube Spam,1956,1713,243
3,Enron Spam,33716,30101,3615


#### Additional Duplicate Check for Enron

Because Enron is substantially larger than the other datasets, its duplicate structure is examined in more detail. To determine whether the detected duplicates are already present in the original data or are introduced by text normalization, exact-text duplicates are compared with duplicates after normalization.

In [10]:
# Compare original and normalized duplicate counts.
print("Exact duplicate texts:", enron["text"].duplicated().sum())

normalized_enron = enron["text"].apply(normalize_text)
print("Normalized duplicate texts:", normalized_enron.duplicated().sum())

Exact duplicate texts: 3222
Normalized duplicate texts: 3615


The results show that 3,222 duplicate messages already exist as exact duplicates in the original Enron data, while 3,615 duplicates are detected after normalization. Normalization therefore introduces 393 additional duplicate rows beyond those that are already exact copies.

These duplicates are not removed at this stage. Their handling will be considered during train/test splitting to prevent the same or equivalent messages from appearing in both sets and causing data leakage.

### 1.4 Cross-Dataset Overlap

In addition to duplicates within each dataset, we check whether the same normalized messages appear across different datasets.

A large overlap would make cross-dataset evaluation less meaningful because the test dataset could contain messages already seen in the training dataset.

In [11]:
from itertools import combinations

# Calculate overlap for every pair of datasets.
overlap_rows = []

for (name1, df1), (name2, df2) in combinations(datasets.items(), 2):
    overlap = get_dataset_overlap(df1, df2)

    overlap_rows.append({
        "dataset_1": name1,
        "dataset_2": name2,
        "overlap": len(overlap),
    })

overlap_df = pd.DataFrame(overlap_rows)
overlap_df

,dataset_1,dataset_2,overlap
0,UCI SMS,Turkish SMS,2
1,UCI SMS,YouTube Spam,1
2,UCI SMS,Enron Spam,0
3,Turkish SMS,YouTube Spam,1
4,Turkish SMS,Enron Spam,0
5,YouTube Spam,Enron Spam,0


### 1.5 Dataset Validation Summary

The datasets differ substantially in both size and class distribution. UCI SMS contains 5,572 messages with a spam ratio of 13.41%, Turkish SMS contains 4,751 messages with a spam ratio of 53.38%, YouTube Spam contains 1,956 comments with a spam ratio of 51.38%, and Enron contains 33,716 emails with a spam ratio of 50.93%. No missing text or label values were found after loading.

Cross-dataset overlap after normalization is very small. Only two messages overlap between UCI SMS and Turkish SMS, one between UCI SMS and YouTube Spam, and one between Turkish SMS and YouTube Spam. No normalized messages from Enron overlap with any of the other datasets.

Internal duplicate analysis revealed 441 duplicate rows in UCI SMS, 55 in Turkish SMS, 243 in YouTube Spam, and 3,615 in Enron after normalization. An additional check showed that 3,222 Enron messages are already exact duplicates before normalization, while normalization introduces 393 additional duplicate rows.

The very low cross-dataset overlap reduces the risk that later transfer experiments are influenced by identical messages appearing in both the training and test datasets.

## 2. Model Development

The project compares five spam detection approaches. Although the models use different text representations and learning algorithms, they follow a common interface so that the same experimental and evaluation code can be reused across all models.

Each model provides:

- `fit(texts, labels)` to train the model.
- `predict_proba(texts)` to produce spam probabilities or comparable prediction scores for evaluation and threshold selection.

The transformer-based models share common functionality for tokenization, batching, device selection, fine-tuning, and prediction through a reusable base implementation in `src/models/transformer_base.py`. Individual transformer model classes specify the pretrained model and its configuration while reusing this common training and inference pipeline.

The model implementations are kept in `src/models/`, while this notebook documents their development, validation, and later experimental comparison.

In [12]:
import numpy as np

# Import the transformer-based spam classifiers.
from src.models.bert_base import BERTBaseModel
from src.models.distilbert import DistilBERTModel
from src.models.xlm_roberta import XLMRoBERTaModel
from src.models.svm import TfidfSVMModel
from src.models.fasttext_model import FastTextModel

### 2.1 BERT Base Uncased

The first transformer model is `bert-base-uncased`, a pretrained English BERT model. It will be fine-tuned for binary spam classification by adding a classification layer with two output classes: ham and spam.

The model receives raw message text, tokenizes it using the corresponding BERT tokenizer, and learns to distinguish between ham and spam during fine-tuning. Its implementation follows the common `fit(texts, labels)` and `predict_proba(texts)` interface defined for all models in this project.

Since transformer fine-tuning is computationally expensive, the full experiments will later be run using a GPU environment.

#### Sequence Length Selection

BERT processes text as sequences of tokens with a fixed maximum length. A larger maximum length preserves more information from long messages but also increases memory usage and computation time.

Since the datasets contain both short SMS messages and longer emails, their token-length distributions are examined before choosing the maximum sequence length.

In [13]:
# Load the pretrained BERT model and tokenizer.
bert_model = BERTBaseModel()

# Measure token lengths without padding or truncation.
length_rows = []

for name, df in datasets.items():
    lengths = [
        len(bert_model.tokenizer.encode(text, add_special_tokens=True))
        for text in df["text"]
    ]

    length_rows.append({
        "dataset": name,
        "median": int(np.median(lengths)),
        "90th_percentile": int(np.percentile(lengths, 90)),
        "95th_percentile": int(np.percentile(lengths, 95)),
        "99th_percentile": int(np.percentile(lengths, 99)),
        "max": int(np.max(lengths)),
    })

length_df = pd.DataFrame(length_rows)
length_df

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] Token i

,dataset,median,90th_percentile,95th_percentile,99th_percentile,max
0,UCI SMS,20,49,55,80,238
1,Turkish SMS,55,98,106,116,206
2,YouTube Spam,15,71,110,175,699
3,Enron Spam,180,737,1107,2219,7688


The token-length distributions differ substantially across the datasets. SMS messages and YouTube comments are generally short, with 99% of their messages remaining below 256 tokens. Enron emails are considerably longer: the median length is 180 tokens and the 90th percentile reaches 737 tokens. Since BERT supports sequences of at most 512 tokens, a substantial portion of the longer Enron emails cannot be represented in full.

A maximum sequence length of 256 tokens is selected as a practical compromise between information retention and computational cost. It covers nearly all messages in the three shorter-text datasets and exceeds the median Enron email length. Longer messages will be truncated during BERT fine-tuning and prediction. Increasing the length to 512 would preserve more Enron content but would increase memory and computation requirements while still truncating many of the longer emails.

#### Implementation Sanity Check

Before using the BERT model in the main experiments, a small sanity check is performed to verify that the complete model pipeline works as expected.

The purpose of this test is not to evaluate classification performance. Instead, it checks whether the model can successfully:

- receive raw text and labels,
- tokenize and batch the messages,
- perform a training step through `fit()`,
- generate spam probabilities through `predict_proba()`.

To keep this functional test lightweight, a very small artificial dataset is used and the model is trained for only one epoch with reduced sequence length and batch size. These temporary settings do not replace the default experiment configuration defined in `src/config.py`.

In [14]:
# Small artificial dataset used only to test the model pipeline.
sanity_texts = [
    "Hey, are we still meeting tomorrow?",
    "Can you send me the lecture notes?",
    "Congratulations! You won a free prize. Claim now!",
    "URGENT! Click here to receive your cash reward!",
]

sanity_labels = [
    "ham",
    "ham",
    "spam",
    "spam",
]

# Use lightweight settings so that the sanity check runs quickly.
sanity_model = BERTBaseModel(
    max_length=64,
    batch_size=2,
    epochs=1,
)

# Verify that the training pipeline runs successfully.
sanity_model.fit(
    sanity_texts,
    sanity_labels,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/1 - Average Loss: 0.7316


In [15]:
# Test inference on two previously unseen example messages.
sanity_test_texts = [
    "Are you coming to class today?",
    "WIN A FREE CASH PRIZE! CLICK NOW!",
]

# Generate the spam probability for each message.
sanity_probabilities = sanity_model.predict_proba(
    sanity_test_texts
)

print("Spam probabilities:", sanity_probabilities)
print("Output shape:", sanity_probabilities.shape)

Spam probabilities: [0.48107925 0.54082286]
Output shape: (2,)


The sanity check completed successfully. The model was able to perform both training and inference without errors, and `predict_proba()` returned one spam probability for each input message, resulting in an output shape of `(2,)`.

The probability values themselves are not interpreted as measures of model performance because the model was fine-tuned using only four artificial messages for a single epoch. The purpose of this test is only to verify that the implementation works correctly before running the actual experiments.

### 2.2 DistilBERT

The second transformer model is DistilBERT, a smaller and faster model derived from BERT through knowledge distillation. DistilBERT retains much of the language understanding capability of BERT while reducing the number of transformer layers and computational requirements.

For spam detection, the pretrained `distilbert-base-uncased` model will be fine-tuned for binary classification using the same ham (`0`) and spam (`1`) label representation as the BERT Base model.

Using DistilBERT provides an important comparison between predictive performance and computational efficiency. This is particularly relevant to the production scenario considered in this project, where a spam filter may need to process millions of messages per day.

The DistilBERT implementation reuses the shared transformer training and inference pipeline defined in `TransformerSpamModel`. Therefore, the main model-specific component is the pretrained `distilbert-base-uncased` architecture, while tokenization, batching, fine-tuning, device handling, and probability prediction follow the same implementation used for BERT Base.

This keeps the transformer implementations consistent and allows differences observed in later experiments to be attributed more clearly to the models rather than to differences in the surrounding training code.

#### Implementation Sanity Check

Before using the DistilBERT model in the main experiments, a small sanity check is performed to verify that the model works correctly with the shared transformer pipeline.

The purpose of this test is not to evaluate classification performance. Instead, it checks whether the model can successfully:

- receive raw text and labels,
- tokenize and batch the messages using the DistilBERT tokenizer,
- perform a training step through `fit()`,
- generate spam probabilities through `predict_proba()`.

To keep this functional test lightweight, a very small artificial dataset is used and the model is trained for only one epoch with reduced sequence length and batch size. These temporary settings do not replace the default DistilBERT configuration defined in `src/config.py`.

In [16]:
# Small artificial dataset used only to test the model pipeline.
distilbert_sanity_texts = [
    "Hey, are we still meeting tomorrow?",
    "Can you send me the lecture notes?",
    "Congratulations! You won a free prize. Claim now!",
    "URGENT! Click here to receive your cash reward!",
]

distilbert_sanity_labels = [
    "ham",
    "ham",
    "spam",
    "spam",
]

# Use lightweight settings so that the sanity check runs quickly.
distilbert_sanity_model = DistilBERTModel(
    max_length=64,
    batch_size=2,
    epochs=1,
)

# Verify that the training pipeline runs successfully.
distilbert_sanity_model.fit(
    distilbert_sanity_texts,
    distilbert_sanity_labels,
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/1 - Average Loss: 0.6613


In [17]:
# Test inference on two previously unseen example messages.
distilbert_sanity_test_texts = [
    "Are you coming to class today?",
    "WIN A FREE CASH PRIZE! CLICK NOW!",
]

# Generate the spam probability for each message.
distilbert_sanity_probabilities = distilbert_sanity_model.predict_proba(
    distilbert_sanity_test_texts
)

print("Spam probabilities:", distilbert_sanity_probabilities)
print("Output shape:", distilbert_sanity_probabilities.shape)

Spam probabilities: [0.55223745 0.5820659 ]
Output shape: (2,)


The DistilBERT sanity check completed successfully. The model was able to use the shared transformer training and prediction pipeline without errors, and `predict_proba()` returned one spam probability for each input message, resulting in an output shape of `(2,)`.

As with the BERT Base sanity check, the probability values are not interpreted as measures of classification performance because the model was fine-tuned using only four artificial messages for a single epoch. This test only verifies that the DistilBERT implementation works correctly before the main experiments.

### 2.3 XLM-RoBERTa

The third transformer model is XLM-RoBERTa Base, a multilingual pretrained language model. Unlike BERT Base and DistilBERT, which use English-only pretrained models, XLM-RoBERTa is designed to represent text across many different languages.

This is particularly relevant to the current project because the datasets contain both English and Turkish messages. Using a multilingual model allows the experiments to examine whether multilingual pretraining provides an advantage when transferring between datasets written in different languages.

The XLM-RoBERTa implementation reuses the shared transformer training and inference pipeline defined in `TransformerSpamModel`. The model-specific component is the pretrained `FacebookAI/xlm-roberta-base` architecture, while tokenization, batching, fine-tuning, device handling, and probability prediction are handled through the same common implementation used for BERT Base and DistilBERT.

Using the same surrounding pipeline and initial training configuration keeps the transformer comparison consistent while allowing the effect of the pretrained model architecture and multilingual representation to be examined.

#### Implementation Sanity Check

Before using the XLM-RoBERTa model in the main experiments, a small sanity check is performed to verify that the model works correctly with the shared transformer pipeline.

The purpose of this test is not to evaluate classification performance. Instead, it checks whether the model can successfully:

- receive raw text and labels,
- tokenize and batch messages containing both English and Turkish text,
- perform a training step through `fit()`,
- generate spam probabilities through `predict_proba()`.

Both English and Turkish examples are included because XLM-RoBERTa is the multilingual transformer used in this project. To keep the test lightweight, the model is trained for only one epoch with reduced sequence length and batch size. These temporary settings do not replace the default XLM-RoBERTa configuration defined in `src/config.py`.

In [18]:
# Small artificial dataset used only to test the model pipeline.
xlm_roberta_sanity_texts = [
    "Hey, are we still meeting tomorrow?",
    "Ders notlarını bana gönderebilir misin?",
    "Congratulations! You won a free prize. Claim now!",
    "TEBRİKLER! Ücretsiz ödül kazandınız. Hemen tıklayın!",
]

xlm_roberta_sanity_labels = [
    "ham",
    "ham",
    "spam",
    "spam",
]

# Use lightweight settings so that the sanity check runs quickly.
xlm_roberta_sanity_model = XLMRoBERTaModel(
    max_length=64,
    batch_size=2,
    epochs=1,
)

# Verify that the training pipeline runs successfully.
xlm_roberta_sanity_model.fit(
    xlm_roberta_sanity_texts,
    xlm_roberta_sanity_labels,
)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: FacebookAI/xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/1 - Average Loss: 0.6408


In [19]:
# Test inference on two previously unseen messages in different languages.
xlm_roberta_sanity_test_texts = [
    "Are you coming to class today?",
    "ÜCRETSİZ HEDİYENİZİ ALMAK İÇİN HEMEN TIKLAYIN!",
]

# Generate the spam probability for each message.
xlm_roberta_sanity_probabilities = (
    xlm_roberta_sanity_model.predict_proba(
        xlm_roberta_sanity_test_texts
    )
)

print(
    "Spam probabilities:",
    xlm_roberta_sanity_probabilities,
)
print(
    "Output shape:",
    xlm_roberta_sanity_probabilities.shape,
)

Spam probabilities: [0.40921527 0.39798868]
Output shape: (2,)


The XLM-RoBERTa sanity check completed successfully. The model was able to use the shared transformer training and prediction pipeline without errors, and `predict_proba()` returned one spam probability for each input message, resulting in an output shape of `(2,)`.

The sanity check also included both English and Turkish messages to verify that multilingual text can pass successfully through the XLM-RoBERTa pipeline. As with the previous transformer sanity checks, the probability values are not interpreted as measures of classification performance because the model was fine-tuned using only four artificial examples for a single epoch.

### 2.4 TF-IDF Character N-grams + Linear SVM

The fourth spam detection approach combines character-level TF-IDF features with a Linear Support Vector Machine (SVM). Unlike the previous transformer models, this approach does not rely on pretrained language representations. Instead, each message is represented as a sparse feature vector constructed directly from its character patterns.

Character-level features are useful for spam detection because they can capture word fragments, unusual spelling, repeated characters, URLs, abbreviations, and other patterns that may occur frequently in spam messages. In this implementation, character n-grams of length 2 to 5 are represented using TF-IDF weights.

The resulting feature vectors are classified using a Linear SVM. Since `LinearSVC` does not directly produce class probabilities, its decision scores are calibrated using sigmoid calibration. This allows the model's `predict_proba()` method to return the estimated probability of the spam class, keeping the output interface consistent with the transformer models.

This approach provides a computationally simpler baseline against which the predictive performance, generalization, speed, and model size of the transformer models can later be compared.

#### Implementation Sanity Check

Before using the TF-IDF + Linear SVM model in the main experiments, a small sanity check is performed to verify that the complete model pipeline works as expected.

The purpose of this test is not to evaluate classification performance. Instead, it checks whether the model can successfully:

- receive raw text and labels,
- learn character-level TF-IDF features from the training messages,
- train and calibrate the Linear SVM classifier through `fit()`,
- generate spam probabilities through `predict_proba()`.

A small artificial dataset containing multiple examples from each class is used because probability calibration requires enough samples to perform cross-validation. The model configuration used in the main experiments remains defined in `src/config.py`.

In [20]:
# Small artificial dataset used only to test the model pipeline.
svm_sanity_texts = [
    "Are we still meeting tomorrow?",
    "Can you send me the lecture notes?",
    "I will call you after class.",
    "Let's have lunch at noon.",
    "The meeting starts at three.",
    "Thanks for your help yesterday.",
    "WIN A FREE CASH PRIZE! CLICK NOW!",
    "Congratulations! You have won a free gift!",
    "URGENT! Claim your cash reward today!",
    "FREE entry! Reply now to collect your prize!",
    "You are our lucky winner. Click here now!",
    "Exclusive offer! Win money instantly!",
]

svm_sanity_labels = [
    "ham",
    "ham",
    "ham",
    "ham",
    "ham",
    "ham",
    "spam",
    "spam",
    "spam",
    "spam",
    "spam",
    "spam",
]

# Create the TF-IDF + Linear SVM model.
svm_sanity_model = TfidfSVMModel()
# Verify that the training pipeline runs successfully.

svm_sanity_model.fit(
    svm_sanity_texts,
    svm_sanity_labels,
)

In [21]:
# Test inference on two previously unseen example messages.
svm_sanity_test_texts = [
    "Are you coming to class today?",
    "WIN A FREE CASH PRIZE! CLICK NOW!",
]

# Generate the spam probability for each message.
svm_sanity_probabilities = svm_sanity_model.predict_proba(
    svm_sanity_test_texts
)

print(
    "Spam probabilities:",
    svm_sanity_probabilities,
)
print(
    "Output shape:",
    svm_sanity_probabilities.shape,
)

Spam probabilities: [0.31701965 0.92328411]
Output shape: (2,)


The TF-IDF + Linear SVM sanity check completed successfully. The model was able to learn character-level TF-IDF features, train the Linear SVM classifier, and perform probability calibration without errors. The `predict_proba()` method returned one spam probability for each input message, resulting in an output shape of `(2,)`.

Unlike the uncalibrated decision scores produced directly by `LinearSVC`, the returned values are calibrated estimates between 0 and 1 and therefore follow the same spam-probability interface used by the transformer models.

The probability values themselves are not interpreted as measures of classification performance because the model was trained using only a small artificial dataset. The purpose of this test is only to verify that the implementation works correctly before running the actual experiments.

### 2.5 fastText

The fifth and final spam detection approach is fastText, a lightweight text classification model designed for efficient training and inference. Unlike the pretrained transformer models used earlier, the fastText model in this project is trained directly on the spam detection datasets rather than being fine-tuned from a pretrained language model.

fastText represents text using learned vector representations and can incorporate word n-grams to capture short local patterns in messages. In this implementation, word n-grams of up to length 2 are used so that the model can learn from both individual words and short phrases such as "free prize" or "claim now".

The model is trained using fastText's supervised classification interface. Since fastText expects training examples in its own labeled text format, the `fit()` method temporarily converts the input messages into lines such as `__label__spam message text` before training the classifier.

The implementation follows the same `fit(texts, labels)` and `predict_proba(texts)` interface used by the other models. The probability assigned to the `spam` label is returned for each input message, allowing fastText to be evaluated using the same metrics and experimental pipeline as the other four approaches.

Because fastText is considerably simpler than transformer-based models, it provides an important comparison of predictive performance against computational efficiency, particularly for the large-scale production scenario considered in this project.

#### Implementation Sanity Check

Before using the fastText model in the main experiments, a small sanity check is performed to verify that the complete model pipeline works as expected.

The purpose of this test is not to evaluate classification performance. Instead, it checks whether the model can successfully:

- receive raw text and labels,
- convert the training examples into the format required by fastText,
- train the supervised classifier through `fit()`,
- generate spam probabilities through `predict_proba()`.

To keep this functional test lightweight, a small artificial dataset is used. The probability values produced by this test are not intended to represent the actual performance of the model.

In [22]:
# Small artificial dataset used only to test the model pipeline.

fasttext_sanity_texts = [
    "Are we still meeting tomorrow?",
    "Can you send me the lecture notes?",
    "I will call you after class.",
    "Let's have lunch at noon.",
    "WIN A FREE CASH PRIZE! CLICK NOW!",
    "Congratulations! You have won a free gift!",
    "URGENT! Claim your cash reward today!",
    "FREE entry! Reply now to collect your prize!",
]

fasttext_sanity_labels = [
    "ham",
    "ham",
    "ham",
    "ham",
    "spam",
    "spam",
    "spam",
    "spam",
]

# Create the fastText model.

fasttext_sanity_model = FastTextModel()

# Verify that the training pipeline runs successfully.

fasttext_sanity_model.fit(
    fasttext_sanity_texts,
    fasttext_sanity_labels,
)

In [23]:
# Test inference on two previously unseen example messages.

fasttext_sanity_test_texts = [
    "Are you coming to class today?",
    "WIN A FREE CASH PRIZE! CLICK NOW!",
]

# Generate the spam probability for each message.

fasttext_sanity_probabilities = (
    fasttext_sanity_model.predict_proba(
        fasttext_sanity_test_texts
    )
)

print(
    "Spam probabilities:",
    fasttext_sanity_probabilities,
)
print(
    "Output shape:",
    fasttext_sanity_probabilities.shape,
)

Spam probabilities: [0.50000876 0.50001395]
Output shape: (2,)


The fastText sanity check completed successfully. The model was able to convert the training examples into the required fastText format, train the supervised classifier, and perform inference without errors. The `predict_proba()` method returned one spam probability for each input message, resulting in an output shape of `(2,)`.

The returned probabilities are valid values between 0 and 1, confirming that the model follows the same spam-probability interface used by the other four approaches. Both probabilities are close to 0.5 because the model was trained using only a very small artificial dataset, so these values are not interpreted as measures of classification performance.

The purpose of this test is only to verify that the fastText implementation works correctly before running the actual experiments.

## 3. Experiment Preparation

Before running the main experiments, a common evaluation setup is prepared so that all models are compared under consistent conditions.

The preparation stage focuses on three important issues:

- preventing duplicate messages from leaking between training and test sets,
- making dataset splits reproducible across repeated runs,
- defining a simple always-ham baseline that always predicts messages as not spam.

The same train/test splits will later be reused across models whenever the experimental setting requires a direct comparison. This ensures that differences in model performance are not caused by different random data splits.

In [24]:
# Import src utilities for experimenting on datasets
from src.experiments import create_train_test_split, always_ham_predictions
from src.metrics import (
    precision_score,
    recall_score,
    f1_score,
    pr_auc_score,
    precision_at_recall,
)

### 3.1 Duplicate-Safe Train/Test Splitting

Randomly splitting individual rows can cause data leakage when duplicate or equivalent messages are present in a dataset. Two messages that become identical after normalization should not be allowed to appear on opposite sides of the train/test split.

To prevent this, normalized message text will be treated as a grouping key. All rows belonging to the same normalized message will therefore be assigned entirely to either the training set or the test set.

Before constructing these splits, duplicate groups are checked for label conflicts. A conflicting group would contain the same normalized message with both `ham` and `spam` labels and would require special handling before evaluation.

In [25]:
# Check whether duplicate messages have conflicting labels.

conflict_rows = []

for name, df in datasets.items():
    check_df = df.copy()

    check_df["normalized_text"] = (
        check_df["text"].map(normalize_text)
    )

    label_counts = (
        check_df
        .groupby("normalized_text")["label"]
        .nunique()
    )

    conflicting_groups = (
        label_counts > 1
    ).sum()

    conflict_rows.append({
        "dataset": name,
        "conflicting_duplicate_groups": conflicting_groups,
    })

conflict_df = pd.DataFrame(conflict_rows)
conflict_df

,dataset,conflicting_duplicate_groups
0,UCI SMS,0
1,Turkish SMS,0
2,YouTube Spam,0
3,Enron Spam,0


#### Duplicate Label Conflict Check

No conflicting duplicate groups were found in any of the four datasets. After normalization, messages that appear multiple times within a dataset consistently retain the same `ham` or `spam` label.

This means that duplicate messages can be treated as groups during train/test splitting without introducing groups that contain contradictory ground-truth labels.

### 3.2 Creating Reproducible Splits

A reusable train/test splitting function is now defined for the same-dataset experiments. Instead of splitting individual rows independently, messages are grouped by their normalized text so that duplicate or equivalent messages cannot appear in both the training and test sets.

The split also aims to preserve the original ham/spam distribution as closely as possible and uses a fixed random seed for reproducibility. The resulting split indices can therefore be reused across different models, ensuring that each model is evaluated on the same training and test messages.

#### Split Validation

Before using the generated splits in the experiments, their properties are checked on the four datasets. In particular, the validation compares the original, training, and test spam ratios and verifies that no normalized message appears in both the training and test sets.

This check helps determine whether the group-based splitting procedure preserves the original row-level class distribution sufficiently well while preventing duplicate leakage.

In [26]:
split_rows = []

for name, df in datasets.items():
    train_df, test_df = create_train_test_split(df)

    train_normalized = set(
        train_df["text"].map(normalize_text)
    )
    test_normalized = set(
        test_df["text"].map(normalize_text)
    )

    split_rows.append({
        "dataset": name,
        "original_size": len(df),
        "train_size": len(train_df),
        "test_size": len(test_df),
        "original_spam_ratio": (df["label"] == "spam").mean(),
        "train_spam_ratio": (train_df["label"] == "spam").mean(),
        "test_spam_ratio": (test_df["label"] == "spam").mean(),
        "normalized_overlap": len(
            train_normalized & test_normalized
        ),
    })

split_check_df = pd.DataFrame(split_rows)
split_check_df

,dataset,original_size,train_size,test_size,original_spam_ratio,train_spam_ratio,test_spam_ratio,normalized_overlap
0,UCI SMS,5572,4458,1114,0.134063,0.134141,0.133752,0
1,Turkish SMS,4751,3801,950,0.533782,0.533807,0.533684,0
2,YouTube Spam,1956,1565,391,0.513804,0.513738,0.514066,0
3,Enron Spam,33716,26973,6743,0.509283,0.509287,0.509269,0


The generated splits preserve the original row-level spam ratios closely across all four datasets while maintaining an approximately 80/20 train/test division. In addition, the normalized overlap between the training and test sets is zero for every dataset, confirming that duplicate or equivalent messages are not split across the two sets.

These results indicate that the group-based splitting procedure provides reproducible and leakage-safe splits without substantially changing the original class distributions.

### 3.3 Metric Implementation Sanity Check

The evaluation metrics were first implemented manually in `src/metrics.py`, as required by the assignment. Before using them in the experiments, their outputs are compared with equivalent scikit-learn calculations on a small example.

This validation checks that the custom precision, recall, F1, PR-AUC, and precision at fixed recall calculations produce the expected results.

In [27]:
from sklearn.metrics import (
    precision_score as sklearn_precision,
    recall_score as sklearn_recall,
    f1_score as sklearn_f1,
    precision_recall_curve as sklearn_pr_curve,
    auc,
)

# Create a small example with labels, predictions, and spam scores.
y_true = [1, 0, 1, 1, 0]
y_pred = [1, 1, 1, 0, 0]
y_score = [0.90, 0.80, 0.60, 0.40, 0.20]

# Compute the scikit-learn precision-recall curve for PR-AUC.
sk_precision, sk_recall, _ = sklearn_pr_curve(
    y_true,
    y_score,
)

# Compare the custom metrics with scikit-learn.
print(
    "Precision:",
    precision_score(y_true, y_pred),
    sklearn_precision(y_true, y_pred),
)

print(
    "Recall:",
    recall_score(y_true, y_pred),
    sklearn_recall(y_true, y_pred),
)

print(
    "F1:",
    f1_score(y_true, y_pred),
    sklearn_f1(y_true, y_pred),
)

print(
    "PR-AUC:",
    pr_auc_score(y_true, y_score),
    auc(sk_recall, sk_precision),
)

Precision: 0.6666666666666666 0.6666666666666666
Recall: 0.6666666666666666 0.6666666666666666
F1: 0.6666666666666666 0.6666666666666666
PR-AUC: 0.7638888888888888 0.7638888888888888


In [28]:
# Compare precision at a fixed recall using the same PR curve.
target_recall = 0.80

sk_valid_precisions = [
    precision
    for precision, recall in zip(sk_precision, sk_recall)
    if recall >= target_recall
]

print(
    "Precision at Recall >= 0.80:",
    precision_at_recall(
        y_true,
        y_score,
        target_recall,
    ),
    max(sk_valid_precisions),
)

Precision at Recall >= 0.80: 0.75 0.75


#### Sanity Check Results

The custom implementations of precision, recall, F1, PR-AUC, and precision at fixed recall produced the same results as their corresponding scikit-learn calculations on the validation example.

This confirms that the manually implemented metrics behave as expected and can be used consistently in the following experiments.

### 3.4 Always-Ham Sanity Baseline

As a simple sanity baseline, every message is predicted as `ham`, regardless of its content. This baseline does not learn from the training data and is not counted as one of the five main models.

Its purpose is to provide a reference point for the experimental results. In particular, it helps reveal cases where a seemingly high accuracy may be caused by class imbalance rather than meaningful spam detection.

In [29]:
# Evaluate the always-ham baseline on each test split.
for dataset_name, df in datasets.items():
    _, test_df = create_train_test_split(df)

    y_true = (
        test_df["label"] == "spam"
    ).astype(int).to_numpy()

    y_pred = always_ham_predictions(len(test_df))

    print(dataset_name)
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1:", f1_score(y_true, y_pred))
    print()

UCI SMS
Precision: 0.0
Recall: 0.0
F1: 0.0

Turkish SMS
Precision: 0.0
Recall: 0.0
F1: 0.0

YouTube Spam
Precision: 0.0
Recall: 0.0
F1: 0.0

Enron Spam
Precision: 0.0
Recall: 0.0
F1: 0.0



#### Baseline Results

As expected, the always-ham baseline obtains zero precision, recall, and F1 on all four datasets because it never predicts the spam class.

These results provide a simple sanity reference for the main model comparisons. The baseline will be included in the experiment result tables but will not be counted as one of the five main models.

## 4. Experiments

The five spam detection approaches are now evaluated using the common datasets, splits, and metrics prepared in the previous sections. The experiments are organized to compare not only in-domain predictive performance, but also cross-dataset generalization, sensitivity to training size, performance under realistic class imbalance, and deployment-related efficiency.

In [30]:
# Import src utilities for experimenting on datasets
from src.experiments import (
    create_train_test_split,
    evaluate_model,
    set_random_seed,
)

### 4.1 Same-Dataset Performance

In the first experiment, each of the five models is trained and evaluated on the same dataset. The duplicate-safe train/test splits created earlier are reused so that all models are compared on consistent data without normalized-message overlap between training and test sets.

The always-ham sanity baseline is also included as a reference but is not counted as one of the five main models. Model performance will be evaluated using precision, recall, F1, PR-AUC, and precision at fixed recall.

#### Preparing the Experiment Splits

The duplicate-safe train/test splitting procedure defined earlier is applied once to each dataset. These same splits will be reused across all five models so that every model is trained and evaluated on exactly the same messages within each dataset.

Reusing fixed splits prevents differences in model performance from being caused by different train/test samples.

In [31]:
# Create and store one fixed train/test split for each dataset.
dataset_splits = {}

for dataset_name, df in datasets.items():
    train_df, test_df = create_train_test_split(df)

    dataset_splits[dataset_name] = (
        train_df,
        test_df,
    )

#### Preparing the Model Set

The five model classes are stored in a common structure so that the same evaluation procedure can later be applied to each approach.

The classes themselves are stored rather than already-created model objects. This allows a fresh model to be created for every training run and ensures that the random seed can be set before model initialization.

In [32]:
# Store the five model classes without creating model instances yet.
model_classes = {
    "BERT Base": BERTBaseModel,
    "DistilBERT": DistilBERTModel,
    "XLM-RoBERTa": XLMRoBERTaModel,
    "TF-IDF + SVM": TfidfSVMModel,
    "fastText": FastTextModel,
}

#### Running the Same-Dataset Experiment

Each model is trained separately on each dataset and evaluated on that dataset's fixed test split. A fresh model instance is created for every model-dataset combination, and the random seed is reset before model initialization.

The random seed is reset before each model is created to make repeated runs as reproducible as possible.

Because transformer training can be expensive and Colab runtimes may disconnect, the result of each completed model-dataset run is saved immediately to Google Drive. If the experiment is restarted, previously completed runs are loaded and skipped instead of being trained again.

In [33]:
# Persistent path for Experiment 1 results.
experiment_1_results_path = (
    "/content/drive/MyDrive/spam-detection/"
    "experiment_1_results.csv"
)

# Load previously completed runs if they exist.
if os.path.exists(experiment_1_results_path):
    experiment_1_results = (
        pd.read_csv(experiment_1_results_path)
        .to_dict("records")
    )
else:
    experiment_1_results = []

# Track completed model-dataset combinations.
completed_runs = {
    (result["dataset"], result["model"])
    for result in experiment_1_results
}

print(
    f"Completed runs found: "
    f"{len(completed_runs)}/"
    f"{len(dataset_splits) * len(model_classes)}"
)

Completed runs found: 0/20


In [ ]:
# Train only model-dataset combinations that have not been completed yet.
for dataset_name, (train_df, test_df) in dataset_splits.items():
    for model_name, model_class in model_classes.items():

        run_key = (dataset_name, model_name)

        # Skip runs that were already saved.
        if run_key in completed_runs:
            print(
                f"Skipping {model_name} on {dataset_name} "
                f"(already completed)."
            )
            continue

        print(f"Running {model_name} on {dataset_name}...")

        # Reset randomness before creating a fresh model.
        set_random_seed()
        model = model_class()

        # Train the model and compute the evaluation metrics.
        results = evaluate_model(
            model,
            train_df,
            test_df,
        )

        # Store the completed run.
        experiment_1_results.append({
            "dataset": dataset_name,
            "model": model_name,
            **results,
        })

        completed_runs.add(run_key)

        # Save immediately so completed work survives runtime disconnects.
        pd.DataFrame(
            experiment_1_results
        ).to_csv(
            experiment_1_results_path,
            index=False,
        )

        print(
            f"Saved {model_name} on {dataset_name}."
        )

        # Release GPU memory before the next model.
        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

Running BERT Base on UCI SMS...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
